# 04 · Does terrain leak into the embeddings?

Clay never sees elevation, slope, or any terrain data -- only 10 Sentinel-2 reflectance bands
plus wavelength/time/location metadata. The Front Range spans ~900m of relief across the AOI
(plains to foothills), so if any embedding axis tracks elevation or slope anyway, that's a real
emergent finding: the model inferring terrain from spectral/textural cues alone.

This also extends the PCA analysis from notebook 03 past the first 3 components -- worth
checking whether a later component (which explains less variance but wasn't examined against
NDVI/NDBI/NDWI) lines up with terrain instead.

**No GPU needed.** This notebook only reads the embeddings and chip metadata already committed
in `docs/data/` -- no Drive mount, no re-embedding. CPU runtime is fine and this should run in
under a minute once the DEM download finishes.

In [ ]:
REPO_URL = "https://github.com/ZanderHirman08/SATEMB.git"

import os

if not os.path.exists("SATEMB"):
    !git clone {REPO_URL}
%cd SATEMB
!pip install -q -r environment/requirements-colab.txt

In [ ]:
import sys

sys.path.append(os.getcwd())

import json

import matplotlib.pyplot as plt
import numpy as np
import odc.stac
from scipy import stats
from sklearn.decomposition import PCA

from src import stac_utils, viz_utils

with open("docs/data/chips.geojson") as f:
    chips_geojson = json.load(f)
features = chips_geojson["features"]

with open("docs/data/embeddings.bin", "rb") as f:
    flat = np.frombuffer(f.read(), dtype="<f4")
embeddings = flat.reshape(len(features), -1)

ndvi = np.array([f["properties"]["ndvi"] for f in features])
clusters = np.array([f["properties"]["cluster"] for f in features])

print(f"{len(features)} chips, {embeddings.shape[1]}-dim embeddings (loaded straight from docs/data/, no Colab-generated files needed)")
os.makedirs("docs/figures", exist_ok=True)

## Fetch elevation over the AOI

[USGS 3DEP Seamless](https://planetarycomputer.microsoft.com/dataset/3dep-seamless), loaded on
the same UTM 13N / 10m grid as the Sentinel-2 mosaic in notebook 01, so chip pixel windows line
up exactly the same way they did for the ESA WorldCover comparison in notebook 03.

In [ ]:
catalog = stac_utils.open_catalog()
dem_items = stac_utils.search_dem(catalog)
print(f"{len(dem_items)} DEM tile(s), gsd={set(it.properties.get('gsd') for it in dem_items)}")

dem_ds = odc.stac.load(
    dem_items, bands=["data"], bbox=stac_utils.FRONT_RANGE_BBOX,
    crs="EPSG:32613", resolution=stac_utils.GSD_M, chunks={"x": 1024, "y": 1024},
)
dem_var = dem_ds["data"]
elevation = dem_var.median(dim="time").compute().values if "time" in dem_var.dims else dem_var.compute().values
elevation = np.where(elevation < -1000, np.nan, elevation)  # 3DEP nodata sentinel
print(f"Elevation grid: {elevation.shape}, range {np.nanmin(elevation):.0f}-{np.nanmax(elevation):.0f} m")

In [ ]:
# Slope in degrees from the elevation raster's finite-difference gradient.
dz_dy, dz_dx = np.gradient(elevation, stac_utils.GSD_M)
slope_deg = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))

plt.figure(figsize=(10, 10))
plt.imshow(elevation, cmap="terrain")
plt.colorbar(label="elevation (m)")
plt.title("Front Range elevation (USGS 3DEP)")
plt.axis("off")
plt.savefig("docs/figures/elevation_map.png", dpi=150, bbox_inches="tight")
plt.show()

## Match elevation/slope to chips

Same pixel-window-by-id matching pattern as the WorldCover comparison in notebook 03.

In [ ]:
grid_by_id = {c["id"]: c for c in stac_utils.make_pixel_chip_grid(*elevation.shape)}

mean_elev, mean_slope, matched = [], [], []
for f in features:
    win = grid_by_id.get(f["properties"]["id"])
    if win is None:
        mean_elev.append(np.nan)
        mean_slope.append(np.nan)
        matched.append(False)
        continue
    mean_elev.append(float(np.nanmean(elevation[win["y_slice"], win["x_slice"]])))
    mean_slope.append(float(np.nanmean(slope_deg[win["y_slice"], win["x_slice"]])))
    matched.append(True)

mean_elev = np.array(mean_elev)
mean_slope = np.array(mean_slope)
matched = np.array(matched)
print(f"Matched {matched.sum()}/{len(features)} chips to elevation data")

## Correlate against PCA components -- extended past the top 3

Notebook 03 only checked whether PC1-PC3 correlate with NDVI/NDBI/NDWI. Here we go further:
compute the top 10 components and check every one against elevation, slope, and NDVI. A later
component that correlates with terrain but explains little variance would be a genuinely new
finding, not something notebook 03 could have caught.

In [ ]:
N_COMPONENTS = 10
pca = PCA(n_components=N_COMPONENTS, random_state=0).fit(embeddings[matched])
pcs_all = np.full((len(features), N_COMPONENTS), np.nan)
pcs_all[matched] = pca.transform(embeddings[matched])

print(f"{'PC':4s} {'var%':>6s} {'r(elev)':>9s} {'r(slope)':>9s} {'r(ndvi)':>9s}")
rows = []
for i in range(N_COMPONENTS):
    r_elev = stats.pearsonr(pcs_all[matched, i], mean_elev[matched])[0]
    r_slope = stats.pearsonr(pcs_all[matched, i], mean_slope[matched])[0]
    r_ndvi = stats.pearsonr(pcs_all[matched, i], ndvi[matched])[0]
    var_pct = pca.explained_variance_ratio_[i] * 100
    rows.append((i + 1, var_pct, r_elev, r_slope, r_ndvi))
    flag = " <-- terrain, not vegetation" if abs(r_elev) > 0.3 and abs(r_ndvi) < 0.2 else ""
    print(f"PC{i+1:<3d}{var_pct:6.1f}% {r_elev:9.2f} {r_slope:9.2f} {r_ndvi:9.2f}{flag}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

var_pcts = [r[1] for r in rows]
r_elevs = [r[2] for r in rows]
axes[0].bar(range(1, N_COMPONENTS + 1), var_pcts, color="#5ec8ff")
axes[0].set_xlabel("PCA component")
axes[0].set_ylabel("% variance explained")
axes[0].set_title("Variance per component")

axes[1].bar(range(1, N_COMPONENTS + 1), r_elevs, color="#f58231")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_xlabel("PCA component")
axes[1].set_ylabel("correlation with elevation (r)")
axes[1].set_title("Elevation correlation per component")

plt.tight_layout()
plt.savefig("docs/figures/pca_elevation_correlation.png", dpi=150)
plt.show()

## Elevation by cluster

A simple, independent sanity check: notebook 03's clusters were built with no elevation
information at all, so if they still separate by mean elevation, that's more evidence the
clusters reflect real physical geography, not just an artifact of the clustering procedure.

In [ ]:
unique_clusters = sorted(set(clusters[matched].tolist()))
cluster_elev_means = [mean_elev[matched & (clusters == c)].mean() for c in unique_clusters]
cluster_elev_stds = [mean_elev[matched & (clusters == c)].std() for c in unique_clusters]

plt.figure(figsize=(8, 5))
plt.bar([str(c) for c in unique_clusters], cluster_elev_means, yerr=cluster_elev_stds, capsize=4, color="#4fd0c4")
plt.xlabel("embedding cluster")
plt.ylabel("mean elevation (m), \u00b1 std")
plt.title("Mean elevation per embedding cluster (clusters never saw elevation)")
plt.tight_layout()
plt.savefig("docs/figures/cluster_elevation.png", dpi=150)
plt.show()

for c, m, s in zip(unique_clusters, cluster_elev_means, cluster_elev_stds):
    print(f"cluster {c}: {m:.0f} m (std {s:.0f} m, n={int((matched & (clusters == c)).sum())})")

## Save elevation/slope onto the live map's data

Adds `elevation` and `slope_deg` properties to the same `docs/data/chips.geojson` the live map
reads, without touching `pca_color`/`cluster`/`ndvi` or embeddings.bin -- purely additive, so the
map keeps working exactly as before. (Wiring up an actual "Elevation" color mode in `docs/app.js`
is a small follow-up if the correlation above turns out to be worth visualizing.)

In [ ]:
extra_props = {
    f["properties"]["id"]: {"elevation": round(e, 1), "slope_deg": round(s, 2)}
    for f, e, s, m in zip(features, mean_elev, mean_slope, matched) if m
}
viz_utils.add_properties_to_geojson("docs/data/chips.geojson", extra_props)

print("\nDone. Commit docs/data/chips.geojson (updated) and the new docs/figures/*.png back to the repo.")